# Milestone 1: Automated Pipeline for arXiv Metadata, TeX Sources, and References

Notebook này chạy quy trình thu thập cho các bài báo trong khoảng `start_id` đến `end_id`. Hãy chạy các ô **từ trên xuống dưới**. Mỗi công đoạn xử lý nằm ở một cell riêng để có thể chạy lại sau khi Colab bị ngắt.

1. **Cào metadata từ arXiv API (Bảng 1):** tìm bài báo hợp lệ trong dải ID, lưu `data/papers.jsonl` và thống kê `data/csv/m1_discovery_stats.csv`.
2. **Tải mã nguồn theo từng phiên bản (Bảng 2):** tự động duyệt qua từng `v1`, `v2`,… của các bài báo đã có trong metadata; chỉ giữ `.tex` và `.bib` trong `data/raw/<id>/vN/`. Trạng thái từng phiên bản nằm trong `versions_manifest.jsonl` và bảng `m1_download_versions.csv` khi xuất snapshot.
3. **Thu thập references từ Semantic Scholar (Bảng 3):** tự động thu thập trích dẫn cho toàn bộ bài báo trong dải ID, lưu cả trích dẫn không có arXiv ID vào `references_raw.jsonl`. Bảng `references.jsonl` cũ chỉ dùng để đối chiếu, không được tự xem là kết quả của bước mới.
4. **Kiểm định và đo hiệu năng (Bảng 4):** kiểm tra dữ liệu bằng hash, tóm tắt độ phủ; benchmark của lượt chạy mới được lưu riêng, không ghi đè bảng benchmark cũ.

### Cơ chế chạy tiếp sau lỗi

Các bước metadata, source và references lưu trạng thái để **resume**: chạy lại sẽ bỏ qua mục đã hoàn tất, còn lỗi có thể được thử lại theo cờ cấu hình. Source và references mặc định chỉ xem kế hoạch, chưa gọi mạng. **Metadata mặc định sẽ gọi arXiv API** cho những ID chưa hoàn tất trong dải đã chọn; dùng `LIMIT` nhỏ để thử trước.

> Dữ liệu lớn nằm trên Google Drive; mã nguồn Git được clone vào ổ cục bộ Colab (`/content`). Không chạy hai phiên cùng ghi vào một thư mục dữ liệu.

## 1. Kết nối Google Drive và Thiết lập Môi trường (trên Colab)

**Mount Drive và lấy mã nguồn.** Chạy đầu tiên trong mỗi runtime Colab. Cell mount Drive, clone hoặc cập nhật đúng nhánh vào `/content/arxiv-code`, rồi trỏ `ARXIV_DATA_DIR` tới thư mục `data` trên Drive. Khi chạy ở máy cá nhân, cell dùng thư mục hiện tại và biến môi trường `ARXIV_DATA_DIR` nếu đã đặt.

Cell này **không yêu cầu** `papers.jsonl` đã tồn tại: bước cào metadata ở phần 4 sẽ tạo file đó cho dải ID mới. Nếu đổi thư mục dữ liệu sau khi đã import `src.config`, hãy khởi động lại runtime để tránh dùng cấu hình cũ.

### Cách sử dụng notebook

1. Trên Colab, chọn **Runtime → Change runtime type → CPU**. Kết nối Google Drive, cài thư viện và đọc S2 Secret ở ba ô đầu. Không đặt key trực tiếp trong notebook.
2. Nhập dải ID được giao và đặt `LIMIT` nhỏ, ví dụ 3, để chạy thử. Cào metadata trước; xem kết quả preflight có bao nhiêu paper hợp lệ và bao nhiêu `vN` cần tải.
3. Bật tải source (`DOWNLOAD_VERSIONS=True`) để tự động tải toàn bộ version của các bài báo trong dải ID đã có từ metadata. Sau đó bật thu raw references (`FETCH_RAW_REFERENCES=True`) để tự động cào trích dẫn cho toàn bộ dải ID. Pipeline tự động lưu trạng thái để resume khi chạy lại; theo dõi số `pending`, `failed`, `pdf_only` và `verified`.
4. Chạy kiểm định sau lượt thu thập cuối cùng, rồi xem tổng kết. Khi báo cáo, phân biệt số liệu **toàn dataset đã xác minh** và **bảng legacy**.

**Dấu hiệu hoàn thành:** không còn phiên bản cần tải hoặc lỗi chưa xử lý, số nguồn đã xác minh phù hợp số phiên bản dự kiến, raw references đã được kiểm tra, báo cáo không còn blocker, và benchmark được đo trên Colab CPU-only với phạm vi đầy đủ. Một lần chạy notebook không tự bảo đảm các điều kiện này.

**Cần làm:** mở notebook trong Colab và chạy ô ngay bên dưới. Nếu Colab hỏi quyền truy cập Drive, chọn tài khoản chứa thư mục dữ liệu rồi cho phép. Kiểm tra hai đường dẫn được in: `CODE_DIR` phải là `/content/arxiv-code`; `DATA_DIR` phải là thư mục `data` trên Drive của nhóm. Nếu dữ liệu nằm nơi khác, đặt `ARXIV_DATA_DIR` trước khi chạy ô này và khởi động lại runtime nếu đã từng import `src.config`.

**Vì sao tách đường dẫn:** Git cập nhật code trên ổ cục bộ Colab, còn paper đã crawl vẫn nằm trên Drive qua các phiên chạy. Ô này chỉ lấy/cập nhật mã nguồn và chọn đường dẫn; chưa crawl paper.

In [ ]:
# KẾT NỐI DRIVE VÀ LẤY MÃ NGUỒN
# Mã nguồn ở /content; dữ liệu tiếp tục nằm trên Google Drive.
import os, sys, subprocess
from pathlib import Path

BRANCH = "feat/optimize-pipeline-and-benchmark"
try:
    from google.colab import drive
except ImportError:
    CODE_DIR = Path.cwd().resolve()
    DATA_DIR = Path(os.environ.get("ARXIV_DATA_DIR", CODE_DIR / "data")).resolve()
else:
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/arxiv-code")
    DATA_DIR = Path(os.environ.get(
        "ARXIV_DATA_DIR", "/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data"
    )).resolve()
    if not (CODE_DIR / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
            "https://github.com/ahtuan171/arXiv-Project.git", str(CODE_DIR),
        ], check=True)
    else:
        actual = subprocess.check_output(
            ["git", "-C", str(CODE_DIR), "branch", "--show-current"], text=True
        ).strip()
        if actual != BRANCH:
            raise RuntimeError("Sai nhánh Git; hãy mở notebook đúng nhánh hoặc dùng runtime mới.")
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.environ["ARXIV_DATA_DIR"] = str(DATA_DIR)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if "src.config" in sys.modules:
    from src import config
    if Path(config.DEFAULT_DATA_DIR).resolve() != DATA_DIR:
        raise RuntimeError("Cấu hình dữ liệu cũ đang dùng; hãy Restart session.")
print("Mã nguồn:", CODE_DIR, "| Dữ liệu:", DATA_DIR)


### 1.1. Cài đặt các thư viện cần thiết

**Cài `requirements.txt`.** Chạy sau khi cell 1 lấy mã nguồn, vì pip cần đọc file requirements từ repository. Chờ lệnh cài đặt hoàn tất rồi mới chạy các cell import module. Khi tạo runtime mới hoặc requirements thay đổi, chạy lại cell này.

**Cần làm:** đợi pip báo hoàn tất. Nếu cài đặt lỗi, sửa môi trường hoặc requirements rồi chạy lại đúng ô này; không chạy tiếp các bước import khi thư viện chưa được cài. Tạo runtime Colab mới cũng cần chạy lại bước cài đặt.


In [ ]:
# CÀI ĐẶT THƯ VIỆN
# Chạy lại cell này sau khi cập nhật requirements.txt hoặc tạo runtime mới.
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(CODE_DIR / "requirements.txt")],
    check=True,
)
print("Đã cài requirements.txt")


### 1.2. Cấu hình Semantic Scholar API Key (tùy chọn)

**Đọc Secret, không kiểm tra key qua mạng.** Trong Colab, mở biểu tượng **chìa khóa** ở thanh bên trái, tạo Secret tên `S2_API_KEY`, dán key rồi bật **Notebook access**. Chạy cell để nạp key vào biến môi trường và đồng bộ cấu hình; kết quả chỉ báo *đã cấu hình* hoặc *chưa có*, không hiện giá trị key.

Không có key vẫn có thể chạy các bước không dùng Semantic Scholar. Bước raw references có thể chịu giới hạn tốc độ của quota không key (3.0s/req); có key sẽ được nâng hạn mức (1.2s/req). Cell này không gửi request thử nên không tiêu tốn quota.

**Cần làm:** nếu có S2 key, mở **Secrets** ở thanh bên Colab, tạo mục đúng tên `S2_API_KEY`, bật quyền truy cập notebook, rồi chạy ô dưới. Dòng `S2 API key: đã cấu hình` xác nhận key đã được nạp thành công vào môi trường và module cấu hình.

In [ ]:
# ĐỌC S2 API KEY TỪ COLAB SECRETS (TÙY CHỌN)
# Tạo S2_API_KEY ở biểu tượng chìa khóa, bật Notebook access. Không in giá trị key.
import os
try:
    from google.colab import userdata
except ImportError:
    pass  # Local: có thể đặt S2_API_KEY bằng biến môi trường hoặc file .env.
else:
    try:
        key = userdata.get("S2_API_KEY")
    except Exception as exc:
        print(f"Lưu ý: Không đọc được Secret 'S2_API_KEY' ({exc})")
        key = None
    if key:
        clean_key = str(key).strip().strip("'\"")
        os.environ["S2_API_KEY"] = clean_key
        try:
            import src.config
            src.config.S2_API_KEY = clean_key
        except Exception:
            pass
    del key

active_key = os.environ.get("S2_API_KEY", "").strip()
if active_key:
    try:
        import src.config
        src.config.S2_API_KEY = active_key
    except Exception:
        pass
    print("S2 API key: đã cấu hình (sẵn sàng sử dụng)")
else:
    print("S2 API key: chưa có (sẽ dùng quota công khai giới hạn)")


## 2. Khai báo khoảng arXiv-ID cần xử lý

Form bên dưới hiện các lựa chọn cấu hình chính. Chạy form sau mỗi lần đổi giá trị; kết quả bên dưới sẽ in số ID được chọn. Những lựa chọn nâng cao được giữ ở giá trị mặc định an toàn trong phần code của cùng ô.

### Các lựa chọn cần biết

| Lựa chọn | Ý nghĩa thường dùng |
| --- | --- |
| `start_id`, `end_id` | Hai đầu của dải ID được phân công; các bước sẽ duyệt qua từng ID này. |
| `LIMIT` | `3` để thử ba ID đầu; `0` để chọn toàn bộ dải. Đây là **số ID chọn**, không phải số paper hợp lệ hay số version tải. |
| `DOWNLOAD_VERSIONS` | Bật tải source thật (`True`) hoặc chỉ xem kế hoạch (`False`). Tự động tải tất cả các version (`v1`, `v2`,...) có trong metadata của từng bài. |
| `FETCH_RAW_REFERENCES` | Bật thu raw references từ Semantic Scholar (`True`) hoặc chỉ xem kế hoạch (`False`). Tự động thu thập references cho toàn bộ các bài trong dải ID. |

### Bốn bước thực hiện theo thứ tự

1. Nhập dải ID của bạn, đặt `LIMIT=3`, giữ hai nút tải ở `False`; chạy form rồi chạy bước metadata và kiểm tra ban đầu. **Metadata tự bật** nên bước này có gọi arXiv và ghi `papers.jsonl`.
2. Nếu metadata hợp lệ, bật `DOWNLOAD_VERSIONS=True`; **chạy lại form**, rồi chạy bước tải source. Pipeline tự động tải toàn bộ version theo metadata. Xem kết quả `success`, `pdf_only`, `failed` và `pending`.
3. Khi sẵn sàng dùng S2, bật `FETCH_RAW_REFERENCES=True`; **chạy lại form**, rồi chạy bước references. Pipeline tự động cào references cho toàn bộ bài báo trong dải ID.
4. Chạy bước kiểm định và tổng kết để xem độ phủ toàn dataset. Sau khi chạy thử ổn, đặt `LIMIT=0` để xử lý trọn vẹn toàn bộ dải ID.

### Các tùy chọn nâng cao (trong phần code của cùng ô)

| Nhu cầu | Giá trị cần đổi | Chú ý |
| --- | --- | --- |
| Muốn lưu bảng version mới | `EXPORT_VERSION_CSV = True` | Form → kiểm định; sau mỗi lượt tải cần xuất lại |
| Muốn xuất JSONL metadata version | `EXPORT_VERSION_METADATA = True` | Form → kiểm định |
| Xuất báo cáo kiểm định | `EXPORT_M1_INTEGRITY_REPORT = True` | Form → kiểm định; tạo `m1_integrity_report.json` |
| Xuất tóm tắt Milestone 1 | `EXPORT_M1_REPORT = True` | Form → tổng kết; tạo `m1_summary.json` |
| Cào lại lỗi mạng cũ | `METADATA_RETRY_ERRORS = True`<br>`VERSION_RETRY_ERRORS = True`<br>`RAW_REF_RETRY_ERRORS = True` | Chỉ bật khi nguyên nhân lỗi (mạng, key) đã được sửa |
| Cào lại HTTP 404 | `METADATA_RETRY_NOT_FOUND = True`<br>`VERSION_RETRY_NOT_FOUND = True`<br>`RAW_REF_RETRY_NOT_FOUND = True` | Thường không cần vì 404 là kết quả chính xác |
| Chỉ đọc lại metadata đã có | `CRAWL_METADATA = False` | Cần `papers.jsonl` đã đầy đủ từ trước |
| Đo đạc benchmark hệ thống | `RUN_BENCHMARK = True` | Bật trước cell 5, chạy đến cell 11 để ghi Bảng 4 |
| Chạy bộ kiểm thử tự động | `RUN_OFFLINE_TESTS = True` | Chạy 12 test offline ở cell 7 trước khi tải source |

`METADATA_BATCH_SIZE=100` là số ID trong mỗi batch arXiv. `AUDIT_SAMPLE_SIZE=10` chỉ giới hạn số ví dụ lỗi được **in ra**, không cắt ngắn dữ liệu được kiểm tra. Nếu chưa hiểu một tùy chọn nâng cao, hãy giữ mặc định.

In [ ]:
#@title Cấu hình dải ID và các bước thu thập { display-mode: "form" }
#@markdown ### 1. Chọn dải ID
#@markdown Nhập ID đầu/cuối được giao. `LIMIT=3` để thử ba ID đầu; `0` chọn cả dải.
start_id = "2502.13235"  #@param {type:"string"}
end_id = "2502.13245"  #@param {type:"string"}
LIMIT = 0  #@param {type:"integer"}

#@markdown ### 2. Tải source LaTeX theo phiên bản
#@markdown Bật sau khi metadata đã chạy. Tự động tải tất cả các version (`v1`, `v2`,...) đã tìm thấy trong metadata cho dải ID.
DOWNLOAD_VERSIONS = False  #@param {type:"boolean"}

#@markdown ### 3. Lấy references từ Semantic Scholar
#@markdown Bật sau khi kiểm tra source. Tự động thu thập references cho toàn bộ các bài trong dải ID.
FETCH_RAW_REFERENCES = False  #@param {type:"boolean"}

#@markdown ### 4. Đo hiệu năng hệ thống (Bảng 4)
#@markdown Bật để đo lường tài nguyên CPU/RAM/Disk và xuất `m1_system_benchmark.csv`.
RUN_BENCHMARK = False  #@param {type:"boolean"}

#@markdown ### 5. Xác thực toàn vẹn cuối cùng (Kiểm định deep hash)
#@markdown Bật để băm SHA-256 toàn bộ source và đối chiếu toàn vẹn sau khi hoàn tất benchmark.
RUN_FINAL_AUDIT = True  #@param {type:"boolean"}

# CẤU HÌNH NÂNG CAO: mặc định an toàn; chỉ sửa khi đã đọc phần hướng dẫn.
# Mở phần code của form để thay đổi các giá trị dưới đây khi cần.
CRAWL_METADATA = True
METADATA_BATCH_SIZE = 100
METADATA_RETRY_ERRORS = True
METADATA_RETRY_NOT_FOUND = False
VERSION_LIMIT = 0  # 0: tự động tải toàn bộ version theo metadata
VERSION_RETRY_ERRORS = True
VERSION_RETRY_NOT_FOUND = False
RAW_REF_LIMIT = 0  # 0: tự động cào references cho toàn bộ paper
RAW_REF_BATCH_SIZE = 100
RAW_REF_RETRY_ERRORS = True
RAW_REF_RETRY_NOT_FOUND = False
RUN_OFFLINE_TESTS = False
EXPORT_VERSION_CSV = False
EXPORT_VERSION_METADATA = False
EXPORT_M1_INTEGRITY_REPORT = False
EXPORT_M1_REPORT = False
AUDIT_SAMPLE_SIZE = 10

from src import config
from src.utils import parse_arxiv_range
id_list = parse_arxiv_range(start_id, end_id)
if LIMIT > 0:
    id_list = id_list[:LIMIT]
if not id_list:
    raise ValueError("Dải ID rỗng")
if METADATA_BATCH_SIZE < 1:
    raise ValueError("METADATA_BATCH_SIZE phải >= 1")
if VERSION_LIMIT < 0 or RAW_REF_LIMIT < 0:
    raise ValueError("Giới hạn lượt tải phải >= 0 (0 = tự động toàn bộ theo metadata)")
if AUDIT_SAMPLE_SIZE < 1:
    raise ValueError("AUDIT_SAMPLE_SIZE phải >= 1")
VALIDATION_REQUIRED = True
print("Số ID đã chọn:", len(id_list), "|", id_list[0], "→", id_list[-1])
print("Metadata:", "crawl/resume" if CRAWL_METADATA else "chỉ đọc dữ liệu có sẵn")
print("Source:", f"tự động tải toàn bộ version theo metadata ({'BẬT' if DOWNLOAD_VERSIONS else 'TẮT'})")
print("References:", f"tự động cào references cho toàn dải ID (Batch: {RAW_REF_BATCH_SIZE}, {'BẬT' if FETCH_RAW_REFERENCES else 'TẮT'})")
print("Benchmark (Bảng 4):", f"đo hiệu năng hệ thống ({'BẬT' if RUN_BENCHMARK else 'TẮT'})")
print("Kiểm định cuối (Deep Audit):", f"{'BẬT' if RUN_FINAL_AUDIT else 'TẮT'}\n")

## 3. Khởi động Bộ theo dõi Benchmark Hệ thống (Bảng 4)

**Bắt đầu đo, chỉ khi `RUN_BENCHMARK=True`.** Tracker lấy mẫu RAM và dung lượng trong khi các bước xử lý chạy; cell 11 sẽ kết thúc và lưu kết quả. Để đo một lượt chạy đầy đủ, bật cờ trước cell này rồi chạy liên tục từ metadata đến kiểm định trên **Colab CPU-only**.

Nếu chỉ chạy thử vài ID hoặc để source/references ở chế độ xem kế hoạch, kết quả đo chỉ phản ánh lượt chạy thử đó. Không dùng nó làm benchmark chính thức cho toàn bộ dải.

**Thứ tự đo:** bật `RUN_BENCHMARK` trong form, chạy ô bắt đầu đo **ngay trước** bước metadata, chạy liên tục các bước xử lý, rồi dừng ở mục 8. Nếu chỉ chạy kiểm định trên dữ liệu đã có, số đo không phải thời gian crawl. Việc bật tracker không tự bật các cờ thu thập.


In [ ]:
# BẮT ĐẦU ĐO LƯỜNG (TÙY CHỌN)
# Bật RUN_BENCHMARK trên Colab CPU-only trước khi chạy các bước thu thập và kiểm định.
from src.benchmark import SystemBenchmarkTracker
tracker = None
if RUN_BENCHMARK:
    tracker = SystemBenchmarkTracker(storage_dirs=[config.DEFAULT_DATA_DIR], disk_sample_interval_sec=0.0)
    tracker.start()
    print("Bắt đầu đo lượt chạy đã chọn; kết thúc ở cell 19.")
else:
    print("Chưa đo benchmark.")


## 4. Bước 1: Cào Metadata bài báo từ arXiv API (Bảng 1: `m1_discovery_stats`)

**Gọi `crawl_papers`.** Đầu vào là danh sách `id_list` từ cell 4. Crawler tìm paper hợp lệ, ghi metadata và danh sách phiên bản vào `data/papers.jsonl`, rồi cập nhật thống kê Bảng 1 trong `data/csv/`. Kết quả in ra số ID đã xử lý, thành công và lỗi để kiểm tra trước khi tải source.

Chạy lại cell này sau khi mất kết nối: chế độ `resume=True` bỏ qua paper đã hoàn tất. Nếu muốn thử lại lỗi đang chờ hoặc ID từng trả 404, bật cờ retry tương ứng ở cell 4 rồi chạy lại cell 6. **Không xóa `papers.jsonl` để retry.**

**Đọc kết quả:** `success` là số paper lấy được metadata trong lượt này; `already_done` là paper được resume bỏ qua; `error`/`not_found` cần xem ledger và retry khi phù hợp. Có paper không tồn tại trong dải ID, vì vậy không lấy độ dài dải làm số paper hợp lệ. `versions` trong metadata là danh sách việc cần tải, chưa chứng minh đã có source.

**Lưu ý:** đây là metadata publication từ arXiv API. Với paper nhiều revision, không suy diễn rằng tiêu đề/tác giả/ngày sửa đổi mới nhất cũng đúng cho mọi revision cũ. Metadata lịch sử chưa xác minh phải được ghi là chưa biết.


In [ ]:
# CÀO METADATA TỪ ARXIV (BẢNG 1)
# Resume bỏ qua ID đã hoàn tất; có thể chạy lại cell này sau lỗi mạng.
from src.metadata_module.crawl_papers import crawl_papers
papers_path = config.DEFAULT_DATA_DIR / "papers.jsonl"
if CRAWL_METADATA:
    metadata_result = crawl_papers(
        id_list=id_list, out_path=papers_path,
        csv_path=config.DEFAULT_DATA_DIR / "csv" / "m1_discovery_stats.csv",
        batch_size=METADATA_BATCH_SIZE, resume=True,
        retry_errors=METADATA_RETRY_ERRORS,
        retry_not_found=METADATA_RETRY_NOT_FOUND,
    )
    VALIDATION_REQUIRED = True
    print("Bảng 1 | metadata:", metadata_result)
else:
    if not papers_path.is_file():
        raise FileNotFoundError("Chưa có papers.jsonl; bật CRAWL_METADATA và chạy lại cell 6.")
    print("Đang dùng metadata đã có:", papers_path)


### 4.1. Kiểm tra metadata và số phiên bản cần tải

**Preflight, chỉ đọc dataset.** `audit_missing_versions` so sánh ID đã chọn với metadata, in số paper/phiên bản dự kiến và mẫu cảnh báo. Kiểm tra các con số này trước khi bật tải source: số phiên bản là **khối lượng cần xử lý**, chưa có nghĩa là file đã tải thành công.

Nếu bật `RUN_OFFLINE_TESTS`, cell còn chạy bộ pytest trên dữ liệu giả. Test đạt chỉ chứng minh logic offline; không xác nhận các file trên Drive đã đủ hoặc API đang hoạt động.

**Đọc kết quả:** kiểm tra số metadata hợp lệ, tổng version dự kiến và các mẫu cảnh báo. Nếu số paper thấp bất thường, quay lại bước metadata thay vì tiếp tục tải source. Preflight chỉ xem metadata và manifest, không hash toàn bộ file; kiểm định đầy đủ nằm ở mục 7.


In [ ]:
# KIỂM TRA METADATA VÀ TEST OFFLINE
# Không gửi request crawl hoặc ghi dữ liệu Drive.
from src.scripts.audit_missing_versions import audit_missing_versions

preflight = audit_missing_versions(
    config.DEFAULT_DATA_DIR, ids=id_list, sample_size=AUDIT_SAMPLE_SIZE
)
print("=== KIỂM TRA METADATA (CHỈ ĐỌC) ===")
for key, val in preflight["summary"].items():
    print(f"{key:31}: {val}")
for warning in preflight.get("warnings", [])[:AUDIT_SAMPLE_SIZE]:
    print("Cảnh báo metadata:", warning)

if RUN_OFFLINE_TESTS:
    OFFLINE_TESTS = [
        "tests/test_version_downloader.py",
        "tests/test_audit_version_downloads.py",
        "tests/test_audit_legacy_matches.py",
        "tests/test_audit_progress.py",
        "tests/test_build_version_metadata.py",
        "tests/test_raw_reference_collector.py",
        "tests/test_audit_m1_integrity.py",
        "tests/test_m1_offline_end_to_end.py",
        "tests/test_build_m1_report.py",
        "tests/test_notebook_layout.py",
        "tests/test_notebook_flow.py",
        "tests/test_notebook_presentation.py",
    ]
    assert all((CODE_DIR / path).is_file() for path in OFFLINE_TESTS)
    child_env = dict(os.environ, ARXIV_DATA_DIR="/tmp/m1-fixtures-only",
                     S2_API_KEY="", PYTHONDONTWRITEBYTECODE="1",
                     PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", *OFFLINE_TESTS],
        cwd=str(CODE_DIR), env=child_env, check=True,
    )
    print("✅ Bộ kiểm thử offline đã đạt.")
else:
    print("Đã bỏ qua kiểm thử offline.")


## 5. Bước 2: Tải và Giải nén File Nguồn TeX theo từng phiên bản (Bảng 2)

**`run_version_downloads`.** Từ metadata đã có, cell lập danh sách `v1`, `v2`,… cho từng paper. Với `DOWNLOAD_VERSIONS=False`, nó chỉ in kế hoạch. Khi bật True, pipeline tự động tải tất cả các version của từng bài báo trong dải ID đã có từ metadata và lưu `.tex`/`.bib` trong `data/raw/<id>/vN/`, cùng marker/hash và lịch sử trạng thái trong `versions_manifest.jsonl`.

Chạy lại để tiếp tục các version còn thiếu; version có marker và hash hợp lệ được tự động bỏ qua (resume). HTTP 404, PDF-only và lỗi mạng được phân biệt. Bảng CSV theo version **chưa tự cập nhật sau mỗi batch**; bật `EXPORT_VERSION_CSV` rồi chạy cell 10 để tạo snapshot mới.

**Đọc kết quả:** `plan.summary` cho biết version nào sẽ thử; `summary` là kết quả lượt chạy hiện tại. `success` cần được kiểm tra bằng marker/hash ở mục 7; `pdf_only` là version không có TeX/Bib, vẫn cần ghi nhận thay vì bỏ paper. `not_found` và `failed` không được tính là source đã hoàn tất. Để biết **độ phủ toàn dải**, xem kiểm định và tổng kết, không chỉ nhìn số lượt tải của một batch.

Đề bài yêu cầu giữ nguyên nội dung TeX và chỉ loại hình bằng cách giải nén `.tex`/`.bib`; không sửa lệnh `\includegraphics` trong file TeX. File source legacy ở `data/raw/<id>/` không được tự coi là `v1` khi chưa xác minh provenance.

**Log tiến độ:** mỗi dòng `[Source]` cho biết ID/version vừa ghi trạng thái, `đã thử/tổng lượt thử (%)` và kết quả.

In [ ]:
# TẢI SOURCE THEO PHIÊN BẢN
# Tự động tải toàn bộ version theo metadata; mỗi log tiến độ là một lượt thử đã ghi vào manifest.
from src.download_module.version_downloader import run_version_downloads
from src.scripts.audit_version_downloads import sync_download_stats_from_manifest

if DOWNLOAD_VERSIONS:
    VALIDATION_REQUIRED = True
    print("Bắt đầu tải source tự động theo metadata; mỗi dòng sẽ cho biết ID/version, trạng thái và % tiến độ.", flush=True)

def on_source_progress(done, total, event):
    pct = 100 * done / total if total else 100
    paper = event["arxiv_id"]
    version = event["version"]
    status = event["status"]
    print(f"[Source] {done}/{total} ({pct:.1f}%) | {paper}/{version} | {status}", flush=True)

version_result = run_version_downloads(
    config.DEFAULT_DATA_DIR, ids=id_list,
    max_downloads=VERSION_LIMIT, dry_run=not DOWNLOAD_VERSIONS,
    retry_errors=VERSION_RETRY_ERRORS,
    retry_not_found=VERSION_RETRY_NOT_FOUND,
    check_hashes=False,
    progress_callback=on_source_progress if DOWNLOAD_VERSIONS else None,
)
print("=== BẢNG 2 | SOURCE THEO VERSION ===")
print("Kế hoạch:", version_result["plan"]["summary"])
print("Kết quả:", version_result["summary"])
if DOWNLOAD_VERSIONS and (config.DEFAULT_DATA_DIR / "versions_manifest.jsonl").is_file():
    dl_csv, v_csv = sync_download_stats_from_manifest(config.DEFAULT_DATA_DIR)
    if dl_csv:
        print(f"Đã cập nhật snapshot Bảng 2: {dl_csv}")
if not DOWNLOAD_VERSIONS:
    print("Chỉ xem kế hoạch, chưa tải source.")

## 6. Bước 3: Thu thập Danh sách Trích dẫn từ Semantic Scholar (Bảng 3)

**`collect_raw_references`.** Với `FETCH_RAW_REFERENCES=False`, cell chỉ xem kế hoạch. Khi bật True, pipeline tự động truy vấn Semantic Scholar cho toàn bộ các bài báo trong dải ID, theo dõi phân trang và giữ cả trích dẫn thiếu arXiv ID. Dữ liệu mới nằm trong `references_raw.jsonl` cùng ledger; `references.jsonl` và CSV references cũ không bị thay thế bởi bước này.

**Tự động ghi snapshot Bảng 3:** Khi hoàn tất hoặc đồng bộ dữ liệu, collector tự động xuất snapshot ra `data/csv/m1_reference_stats.csv` theo cấu trúc chuẩn Milestone 1 (`arxiv_id`, `ref_status`, `num_crawled_refs`, `num_arxiv_refs`, `ref_time_sec`) phục vụ kiểm tra và báo cáo.

Kết quả S2 là thông tin ở **cấp publication**; không đủ bằng chứng để gán một trích dẫn cho `v1` hay `v2`. Nếu API trả 429 hoặc lỗi mạng, xem trạng thái in ra rồi chạy lại; chỉ bật cờ retry khi muốn thử lại lỗi ngay.

**Đọc kết quả:** số paper truy vấn thành công khác với số dòng reference; mỗi paper có thể có nhiều trích dẫn và nhiều trang API. Collector chỉ công nhận paper hoàn tất sau khi lấy đủ trang. Trích dẫn không có arXiv ID vẫn được giữ ở raw dataset; chỉ mục có arXiv ID phù hợp mới có thể góp vào bảng `references` công bố.

S2 trả thông tin ở cấp publication và có thể không phân biệt các revision. Vì vậy dữ liệu hiện tại chưa chứng minh mỗi trích dẫn thuộc `v1` hay `v2`; phần này phải ghi rõ là chưa biết khi đối chiếu yêu cầu metadata/reference cho từng phiên bản.

**Log tiến độ:** mỗi dòng `[References]` cho biết paper vừa ghi ledger, số mục raw reference, trạng thái và `% paper` của lượt chạy hiện tại. Số mục bằng 0 có thể là không tìm thấy hoặc lỗi; xem trạng thái bên cạnh.

In [ ]:
# CÀO RAW REFERENCES
# Tự động cào references cho toàn dải ID; mỗi log tiến độ là một paper đã ghi trạng thái vào ledger.
import os
from src.crawl_references_module.raw_reference_collector import (
    collect_raw_references,
    export_raw_reference_stats,
    publish_references,
    prune_unverified_references,
)

# Đồng bộ API key hiện tại từ môi trường sang config nếu có
s2_key = (os.environ.get("S2_API_KEY") or getattr(config, "S2_API_KEY", "") or "").strip()
if s2_key:
    config.S2_API_KEY = s2_key

if FETCH_RAW_REFERENCES:
    VALIDATION_REQUIRED = True
    pruned = prune_unverified_references(config.DEFAULT_DATA_DIR)
    if pruned:
        print(f"Đã tự động làm sạch {len(pruned)} bài bị ghi dở dang/mismatch do lỗi mạng: {pruned}")
    quota_desc = f"Keyed quota ({config.S2_API_MIN_INTERVAL}s/req)" if s2_key else f"Anonymous quota ({config.S2_API_ANONYMOUS_MIN_INTERVAL}s/req)"
    print(f"Bắt đầu hỏi Semantic Scholar (Batch size: {RAW_REF_BATCH_SIZE}, {quota_desc}); mỗi dòng sẽ cho biết paper, số references và % tiến độ.", flush=True)

def on_raw_progress(done, total, event):
    pct = 100 * done / total if total else 100
    paper = event["arxiv_id"]
    status = event["status"]
    count = event.get("raw_refs_total", 0)
    error = event.get("error", "")
    print(f"[References] {done}/{total} ({pct:.1f}%) | {paper} | {status} | {count} mục" +
          (f" | {error}" if error else ""), flush=True)

raw_ref_result = collect_raw_references(
    config.DEFAULT_DATA_DIR, ids=id_list,
    max_papers=RAW_REF_LIMIT,
    batch_size=RAW_REF_BATCH_SIZE,
    dry_run=not FETCH_RAW_REFERENCES,
    retry_errors=RAW_REF_RETRY_ERRORS,
    retry_not_found=RAW_REF_RETRY_NOT_FOUND,
    progress_callback=on_raw_progress if FETCH_RAW_REFERENCES else None,
)
print("=== BẢNG 3 | RAW REFERENCES ===")
print(raw_ref_result["summary"])
if (config.DEFAULT_DATA_DIR / "references_raw.jsonl").is_file():
    publish_references(config.DEFAULT_DATA_DIR)
if raw_ref_result.get("csv_path"):
    print(f"Đã xuất snapshot Bảng 3: {raw_ref_result['csv_path']}")
elif (config.DEFAULT_DATA_DIR / "references_raw_ledger.jsonl").is_file():
    ref_csv = export_raw_reference_stats(config.DEFAULT_DATA_DIR)
    print(f"Đã đồng bộ snapshot Bảng 3 và references.jsonl: {ref_csv}")
if not FETCH_RAW_REFERENCES:
    print("Chưa gọi Semantic Scholar hoặc ghi dữ liệu mới.")


## 7. Hoàn thành Benchmark Hệ thống (Bảng 4)

**Dừng tracker và lưu số đo lượt chạy.** Chỉ khi `RUN_BENCHMARK=True`, cell ghi `data/csv/m1_version_pipeline_benchmark.csv`. File này tách khỏi `m1_system_benchmark.csv` legacy. Kết quả gồm wall time, RAM và storage của **lượt chạy vừa đo**; nếu chạy từng cell cách quãng hoặc chỉ lấy một phần dải, hãy ghi rõ phạm vi đó khi báo cáo.

Benchmark đo chính xác toàn bộ chu trình cào dữ liệu: từ metadata, tải giải nén source, lược bỏ ảnh, đến cào raw references và xuất các file CSV bảng thống kê.

**Cách dùng số đo:** ghi lại dải ID, giới hạn lượt tải, trạng thái các cờ và runtime CPU-only cùng kết quả CSV. Đề bài còn yêu cầu thời gian trung bình mỗi paper, thời gian discovery, RAM trung bình/đỉnh, disk đỉnh và dung lượng cuối; CSV benchmark lượt chạy hiện **chưa tự chứng minh toàn bộ các chỉ số và phạm vi này**. Bảng `m1_system_benchmark.csv` cũ không thay thế được phép đo của pipeline `vN`.

In [ ]:
# KẾT THÚC ĐO LƯỜNG LƯỢT CHẠY
# CSV riêng cho pipeline mới; chỉ ghi khi RUN_BENCHMARK=True.
if RUN_BENCHMARK:
    if tracker is None:
        raise RuntimeError("Chạy cell 9 trước khi đo benchmark.")
    benchmark_path = config.DEFAULT_DATA_DIR / "csv" / "m1_version_pipeline_benchmark.csv"
    benchmark_result = tracker.stop(total_papers=len(id_list), csv_path=benchmark_path)
    legacy_bench_path = config.DEFAULT_DATA_DIR / "csv" / "m1_system_benchmark.csv"
    try:
        benchmark_result.to_csv(legacy_bench_path, index=False)
    except Exception:
        pass
    print("Benchmark lượt chạy:", benchmark_path)
    print(benchmark_result.to_string(index=False))
    print("Chỉ dùng làm benchmark toàn dải khi đã chạy đủ các bước trên Colab CPU-only.")
else:
    print("Chưa có benchmark mới; không dùng bảng legacy làm bằng chứng cho vN.")


### Chạy lại và phục hồi các ID bị lỗi

1. **Runtime Colab bị ngắt:** chạy lại cell 1–4, rồi chạy cell thu thập cần tiếp tục. Giữ nguyên dữ liệu Drive.
2. **Metadata thiếu/lỗi:** xem Bảng 1 và cảnh báo cell 7; chỉnh cờ retry metadata ở cell 4, chạy lại cell 6–7.
3. **Source hoặc S2 lỗi:** chỉnh cờ retry tương ứng ở cell 4, chạy lại cell 8 hoặc 9. Các giới hạn lượt thử vẫn có hiệu lực mỗi lần chạy.
4. **Dữ liệu đã thay đổi:** chạy lại cell kiểm định trước cell tổng kết để bảng tổng kết dùng kiểm định mới nhất.

## 8. Kiểm định Source, Metadata và References (Tùy chọn)

**Audit toàn dataset.** Được kiểm soát bởi cờ `RUN_FINAL_AUDIT`. Cell xác minh marker/hash của từng source `vN`, xây chỉ mục metadata phiên bản và kiểm tra tính nhất quán của references. Audit toàn bộ thư mục dữ liệu, **không chỉ dải ID ở cell 4**, nên có thể mất thời gian trên Drive. `needs_fix` nghĩa là có lỗi cấu trúc cần xem; `in_progress` nghĩa là còn công việc chưa hoàn tất.

Mặc định cell chỉ in kết quả khi `RUN_FINAL_AUDIT=True`. Các cờ `EXPORT_VERSION_CSV`, `EXPORT_VERSION_METADATA`, `EXPORT_M1_INTEGRITY_REPORT` mới ghi các snapshot tương ứng. Sau bất kỳ lượt crawl mới nào, chạy lại cell này trước khi xem tổng kết.

**Đọc kết quả:** `verified_success`/`verified_pdf_only` là kết quả đã kiểm chứng; `pending`, `failed`, `not_found`, `conflicts`, `missing_files` là các trạng thái cần đánh giá tiếp. `needs_fix` chỉ ra lỗi dữ liệu; `in_progress` chỉ ra phần chưa hoàn tất; trạng thái nhất quán cấu trúc cũng chưa chứng minh metadata lịch sử từng revision đã đầy đủ.

Nếu file references cũ bị hỏng, audit có thể báo blocker dù bước raw references mới chạy thành công. Xử lý nguồn dữ liệu hỏng có kiểm chứng trước khi tuyên bố hoàn thành; không ghi đè bảng cũ từ notebook này.

**Log kiểm định:** in 5 version đầu, mỗi 100 version, version cuối và mọi mục lỗi quan trọng. Phần trăm ở đây là tỷ lệ **đã kiểm tra**, không phải tỷ lệ dữ liệu đạt. Kết quả tổng hợp bên dưới mới cho biết độ phủ và blocker.

In [ ]:
# KIỂM ĐỊNH SOURCE, METADATA VÀ REFERENCES
# In mẫu đầu, mỗi 100 version và các lỗi; tránh hàng nghìn dòng pending trên Drive.
from src.scripts.audit_version_downloads import (
    audit_version_downloads,
    export_version_csv,
    export_download_stats_csv,
)
from src.scripts.build_version_metadata import build_version_metadata, export_version_metadata
from src.scripts.audit_m1_integrity import audit_m1_integrity, export_integrity_report
from src.crawl_references_module.raw_reference_collector import (
    export_raw_reference_stats,
    publish_references,
)

if RUN_FINAL_AUDIT:
    print("Bắt đầu xác minh marker/hash của toàn bộ version trong dataset...", flush=True)
    def on_verify(done, total, record):
        status = record["status"]
        if done <= 5 or done % 100 == 0 or done == total or status in {"conflict", "missing_files", "failed"}:
            pct = 100 * done / total if total else 100
            paper = record["arxiv_id"]
            version = record["version"]
            print(f"[Audit] {done}/{total} ({pct:.1f}%) | {paper}/{version} | {status}", flush=True)

    version_audit = audit_version_downloads(
        config.DEFAULT_DATA_DIR, sample_size=AUDIT_SAMPLE_SIZE,
        progress_callback=on_verify,
    )
    print("Xong kiểm tra source; đang lập chỉ mục metadata theo version...", flush=True)
    metadata_report = build_version_metadata(
        config.DEFAULT_DATA_DIR, sample_size=AUDIT_SAMPLE_SIZE,
    )

    # Tự động xuất snapshot Bảng 2 và Metadata versions nếu có dữ liệu hoặc được yêu cầu
    if EXPORT_VERSION_CSV or version_audit["summary"]["expected_versions"] > 0:
        print("Bảng version CSV:", export_version_csv(version_audit))
        print("Bảng download stats (Bảng 2):", export_download_stats_csv(version_audit))
    if EXPORT_VERSION_METADATA or len(metadata_report["rows"]) > 0:
        print("Metadata theo version:", export_version_metadata(metadata_report))

    # Đồng bộ snapshot Bảng 3 và xuất file trích dẫn công bố references.jsonl
    if (config.DEFAULT_DATA_DIR / "references_raw.jsonl").is_file():
        print("Published references:", publish_references(config.DEFAULT_DATA_DIR))
    if (config.DEFAULT_DATA_DIR / "references_raw_ledger.jsonl").is_file():
        print("Snapshot Bảng 3 references:", export_raw_reference_stats(config.DEFAULT_DATA_DIR))

    print("Đang đối chiếu metadata, source và references toàn dataset...", flush=True)
    integrity_report = audit_m1_integrity(
        config.DEFAULT_DATA_DIR, sample_size=AUDIT_SAMPLE_SIZE,
        version_report=version_audit, metadata_report=metadata_report,
    )
    VALIDATION_REQUIRED = False
    print("=== TRẠNG THÁI KIỂM ĐỊNH:", integrity_report["status"], "===")
    print("Bao phủ:", integrity_report["coverage"])
    print("Source:", integrity_report["source_versions"])
    print("Raw references:", integrity_report["raw_references"])
    for label in ("blockers", "incomplete", "warnings"):
        for item in integrity_report[label][:AUDIT_SAMPLE_SIZE]:
            print(label.upper(), ":", item)
    if EXPORT_M1_INTEGRITY_REPORT:
        print("Báo cáo kiểm định:", export_integrity_report(integrity_report))
    else:
        print("Đã kiểm định, chưa xuất báo cáo ra Drive.")
else:
    print("Đã bỏ qua bước kiểm định xác thực toàn vẹn (RUN_FINAL_AUDIT = False).")
    print("Các bảng thống kê snapshot đã được xuất trực tiếp trong các bước tải.")

## 9. Kiểm tra và Trích xuất các Bảng Thống kê Milestone 1

**Tổng kết.** Cell dùng kết quả kiểm định ở cell 10 để hiển thị số paper hợp lệ, số version dự kiến, số source đã xác minh, độ phủ và trạng thái raw references. Nếu dữ liệu đã thay đổi từ lần audit cuối, cell yêu cầu chạy lại cell 10. `EXPORT_M1_REPORT=True` mới lưu báo cáo JSON phục vụ Jupyter Book.

Bên dưới là cách xem bốn bảng quen thuộc từ notebook `main`: Bảng 1 metadata (có thể gồm lượt crawl mới), Bảng 2 và 3 legacy, Bảng 4 benchmark legacy. Bảng source theo `vN` và benchmark mới được hiển thị **riêng**. Đừng cộng hoặc thay thế các số legacy bằng số version mới; báo cáo cũng chỉ rõ số liệu/bằng chứng nào chưa có.

**Đọc bảng:** so `expected_versions` với `verified_source_versions` và xem danh sách `missing_evidence`. Chỉ khi snapshot CSV version còn mới mới dùng số đo kích thước/độ phủ từ snapshot. Bảng 1 metadata có thể chứa lượt crawl mới; Bảng 2–4 legacy được gắn nhãn để tránh nhầm với kết quả `vN`.

**Trước khi nộp:** kiểm tra độ phủ toàn bộ dải ID được giao, các mục lỗi còn lại, metadata/reference cần cho từng revision, thống kê corpus và benchmark chính thức. Báo cáo Jupyter Book phải sinh bảng/hình khi build và được đặt ở **repository báo cáo công khai riêng** theo đặc tả môn học; notebook này không tự hoàn thành các yêu cầu nộp bài đó.


In [ ]:
# TỔNG HỢP CÁC BẢNG THỐNG KÊ VÀ BÁO CÁO
# Dùng kết quả đã xác minh từ cell kiểm định hoặc đọc trực tiếp từ các file CSV.
from src.scripts.build_m1_report import build_m1_report, export_m1_report
from IPython.display import display
import pandas as pd

if RUN_FINAL_AUDIT:
    if "integrity_report" not in globals() or VALIDATION_REQUIRED:
        raise RuntimeError("Sau khi dữ liệu thay đổi, phải chạy lại cell kiểm định xác thực.")
    m1_summary = build_m1_report(
        config.DEFAULT_DATA_DIR, integrity_report=integrity_report,
    )
    c = m1_summary["coverage"]
    display(pd.DataFrame([
        {"Metric": "Valid metadata papers", "Value": c["valid_metadata_papers"]},
        {"Metric": "Expected source versions", "Value": c["expected_versions"]},
        {"Metric": "Verified source versions", "Value": c["verified_source_versions"]},
        {"Metric": "Version coverage (%)", "Value": c["source_coverage_pct"]},
        {"Metric": "Raw reference papers verified", "Value": c["raw_reference_papers_verified"]},
    ]))
    print("Trạng thái dữ liệu:", m1_summary["integrity_status"])
    print("Kích thước source:", m1_summary["source_file_statistics"]
          or m1_summary["source_statistics_unavailable_reason"])
    print("Benchmark pipeline mới:",
          m1_summary["benchmark_evidence"]["official_version_pipeline_metrics"])
    print("Các bằng chứng còn thiếu:")
    for item in m1_summary["missing_evidence"]:
        print(" -", item)
    if EXPORT_M1_REPORT:
        print("Báo cáo tóm tắt:", export_m1_report(m1_summary))
    else:
        print("✅ Chỉ xem thống kê, chưa xuất file mới.")
else:
    print("ℹ️ Bỏ qua báo cáo kiểm định chuyên sâu (RUN_FINAL_AUDIT = False).")
    print("Hiển thị 4 bảng thống kê trực tiếp từ các file CSV đã thu thập:")

# Giữ cách trình bày 4 bảng như notebook nhánh main.
# Bảng legacy chỉ để xem/đối chiếu; không được coi là số liệu vN mới.
print("\n" + "=" * 65)
print("4 BẢNG THỐNG KÊ MILESTONE 1 (ĐỊNH DẠNG NOTEBOOK GỐC)")
print("=" * 65)
legacy_tables = [
    ("Bảng 1 — Khám phá metadata (gồm lượt chạy mới)", "m1_discovery_stats.csv"),
    ("Bảng 2 — Tải source cấp paper (legacy)", "m1_download_stats.csv"),
    ("Bảng 3 — Trích dẫn cấp paper (legacy)", "m1_reference_stats.csv"),
    ("Bảng 4 — Benchmark pipeline cũ (không phải benchmark vN)", "m1_system_benchmark.csv"),
]
for label, filename in legacy_tables:
    path = config.DEFAULT_DATA_DIR / "csv" / filename
    print(f"\n--- {label} ---")
    if not path.is_file():
        print("Chưa có bảng:", path)
        continue
    try:
        table = pd.read_csv(path, dtype={"arxiv_id": str})
        print("Tổng dòng:", len(table))
        display(table.head(10))
    except Exception as exc:
        print("Không thể đọc bảng:", filename, "|", exc)

version_path = config.DEFAULT_DATA_DIR / "csv" / "m1_download_versions.csv"
print("\n--- Bảng 2 bổ sung: tải source theo từng version ---")
if version_path.is_file():
    try:
        display(pd.read_csv(version_path, dtype={"arxiv_id": str}).head(10))
        if "integrity_report" in globals() and not integrity_report.get("snapshots", {}).get("download_versions_csv_current", False):
            print("⚠️ Bảng version đã cũ. Hãy xuất lại từ cell kiểm định.")
    except Exception as exc:
        print("Không thể đọc version CSV:", exc)
else:
    print("Chưa xuất version CSV. Bật EXPORT_VERSION_CSV rồi chạy lại cell kiểm định.")

new_benchmark_path = config.DEFAULT_DATA_DIR / "csv" / "m1_version_pipeline_benchmark.csv"
if new_benchmark_path.is_file():
    print("\n--- Benchmark pipeline vN (phạm vi lượt chạy) ---")
    display(pd.read_csv(new_benchmark_path).tail(5))